In [14]:
import subprocess

min_heap = 8192
max_heap = 8192

repetitions = 16

renaissance = 'renaissance-gpl-0.16.1.jar'
ren_options = []
# ren_options += [ '-r', repetitions ]
csv_res_dir = 'res_csv'
stdout_res_dir = 'res_stdout'
stderr_res_dir = 'res_stderr'

def get_stdout(version: str, bench: str):
    return f'{stdout_res_dir}/{version}/{bench}'

def get_stderr(version: str, bench: str):
    return f'{stderr_res_dir}/{version}/{bench}'

def get_csv(version: str, bench: str):
    return f'{csv_res_dir}/{version}/{bench}'

mem_track = 'summary'
java_options = []
java_options += [ '-XX:+UnlockDiagnosticVMOptions' ]
java_options += [ f'-Xms{min_heap}m', f'-Xmx{max_heap}m' ]
java_options += [ f'-XX:NativeMemoryTracking={mem_track}', '-XX:+PrintNMTStatistics' ]
java_options += [ '-XX:+PrintMethodData' ]

mem_plugin = '../../plugin-jmxmemory-assembly-0.0.1.jar'
plugin = []
# plugin += [ '--plugin', mem_plugin ]

bench_list_proc = subprocess.Popen(['java', '-jar', renaissance, '--raw-list'], stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True)
stdout: str = bench_list_proc.communicate()[0]
bench_list = stdout.splitlines()
bench_list.sort()
# bench_list.remove('db-shootout')
bench_count = len(bench_list)


upstream = 'upstream'
latest = 'latest'


In [15]:
import subprocess

force_run = True
def run(version: str, bench: str):
    try:
        open(get_stdout(version, bench))
        if not force_run:
            return
    except:
        pass

    print(f'running {bench} {version}...')
    jdk_path = f'../build/{version}/images/jdk/bin'
    java_path = f'{jdk_path}/java'
    java_cmd = [ java_path] + java_options + [ '-jar', renaissance, bench ] + ren_options
    java_cmd += plugin
    java_cmd += [ '--csv', get_csv(version, bench) ]
    java_cmd = [str(v) for v in java_cmd]

    with open(get_stdout(version, bench), 'w') as f_out:
        with open(get_stderr(version, bench), 'w') as f_err:
            bench_proc = subprocess.Popen(java_cmd, stdout=f_out, stderr=f_err, text=True)
            bench_proc.wait()
            print(f'{bench} {version} exited with code {bench_proc.returncode}')

for bench in ["rx-scrabble"]:
    run(upstream, bench)
    run(latest, bench)


running rx-scrabble upstream...
rx-scrabble upstream exited with code 0
running rx-scrabble latest...
rx-scrabble latest exited with code 0


In [16]:
from __future__ import annotations
from dataclasses import dataclass, field, asdict
from typing import List, Optional, get_origin, get_args
import json
import re
import csv
import numpy as np

@dataclass
class BenchResult:
    version: str
    bench: str
    total_mdo_size: int
    total_special_mdo_size: int
    total_special_mdos: int
    total_bytecodes: int
    total_invokes: int
    time_median_ns: int
    time_95_percentile_ns: int
    time_99_percentile_ns: int
    memory_tracking: dict[str, int] = field(default_factory=dict[str, int])
    invokes: dict[str, int] = field(default_factory=dict[str, int])
    invokes_mdo_sizes: dict[str, int] = field(default_factory=dict[str, int])

import typing
from dataclasses import is_dataclass
from typing import get_origin, get_args

def resolve_type(type_, globalns):
    if isinstance(type_, str):
        return eval(type_, globalns)
    return type_

def from_dict(cls, data, *, globalns=None):
    if not is_dataclass(cls):
        return data  # Primitive type or unsupported

    if not isinstance(data, dict):
        raise ValueError(f"Expected dict for {cls.__name__}, got {type(data).__name__}")

    if globalns is None:
        globalns = globals()

    kwargs = {}
    for field in cls.__dataclass_fields__.values():
        field_name = field.name
        field_type = resolve_type(field.type, globalns)
        raw_value = data.get(field_name)

        origin = get_origin(field_type)
        args = get_args(field_type)

        if raw_value is None:
            kwargs[field_name] = None

        elif origin in {list, typing.List}:
            item_type = resolve_type(args[0], globalns)
            kwargs[field_name] = [
                from_dict(item_type, item, globalns=globalns)
                if is_dataclass(item_type) else item
                for item in raw_value
            ]

        elif origin in {dict, typing.Dict}:
            key_type, val_type = map(lambda t: resolve_type(t, globalns), args)
            if key_type != str:
                raise NotImplementedError("Only Dict[str, T] supported")
            kwargs[field_name] = {
                k: from_dict(val_type, v, globalns=globalns)
                if is_dataclass(val_type) else v
                for k, v in raw_value.items()
            }

        elif origin is typing.Union and type(None) in args:  # Optional[T]
            inner_type = resolve_type(args[0], globalns)
            kwargs[field_name] = (
                from_dict(inner_type, raw_value, globalns=globalns)
                if raw_value is not None else None
            )

        elif is_dataclass(field_type):
            kwargs[field_name] = from_dict(field_type, raw_value, globalns=globalns)

        else:
            kwargs[field_name] = raw_value

    return cls(**kwargs)

@dataclass
class BenchDiff:
    upstream: str
    latest: str
    time_median_ns_diff: int
    time_median_rel_diff: float
    time_95_percentile_ns_diff: int
    time_95_percentile_rel_diff: float
    time_99_percentile_ns_diff: int
    time_99_percentile_rel_diff: float
    memory_tracking_abs_diff: int
    memory_tracking_rel_diff: float
    total_mdo_size_abs_diff: int
    total_mdo_size_rel_diff: float
    total_non_special_mdo_size_abs_diff: int
    total_non_special_mdo_size_rel_diff: float
    
    def __init__(self, upstream: BenchResult, latest: BenchResult):
        self.upstream = upstream
        self.latest = latest
        self.time_median_ns_diff = latest.time_median_ns - upstream.time_median_ns
        self.time_median_rel_diff = (latest.time_median_ns - upstream.time_median_ns) / upstream.time_median_ns
        self.time_95_percentile_ns_diff = latest.time_95_percentile_ns - upstream.time_95_percentile_ns
        self.time_95_percentile_rel_diff = (latest.time_95_percentile_ns - upstream.time_95_percentile_ns) / upstream.time_95_percentile_ns
        self.time_99_percentile_ns_diff = latest.time_99_percentile_ns - upstream.time_99_percentile_ns
        self.time_99_percentile_rel_diff = (latest.time_99_percentile_ns - upstream.time_99_percentile_ns) / upstream.time_99_percentile_ns
        self.memory_tracking_abs_diff = { ku: (vl - vu) for (ku, vu), (_, vl) in zip(upstream.memory_tracking.items(), latest.memory_tracking.items()) }
        self.memory_tracking_rel_diff = { ku: (vl / vu) for (ku, vu), (_, vl) in zip(upstream.memory_tracking.items(), latest.memory_tracking.items()) }
        self.total_mdo_size_abs_diff = latest.total_mdo_size - upstream.total_mdo_size
        self.total_mdo_size_rel_diff = latest.total_mdo_size / upstream.total_mdo_size

        self.total_non_special_mdo_size_abs_diff = (latest.total_mdo_size - latest.total_special_mdo_size) - upstream.total_mdo_size
        self.total_non_special_mdo_size_rel_diff = (latest.total_mdo_size - latest.total_special_mdo_size) / upstream.total_mdo_size


top_level_bytecode_pattern = re.compile(r'^\s{0,4}\d+\s[a-z0-9_]+($|\s)')
bytecode_pattern = re.compile(r'^\s*\d+\s[a-z0-9_]+($|\s)')
nmt_pattern = re.compile(r'^-\s*(\S.+) \(reserved=\d+, committed=(\d+)\)')

special_mdo_pattern = re.compile(r'^\s*specialized mdo size: (\d+)')

special_size_pattern = re.compile(r'^\s*specialized data size: (\d+)')

top_level_invoke_pattern = re.compile(r'^\s{0,4}\d+\sinvoke')
invoke_pattern = re.compile(r'^\s*\d+\sinvoke')
invoke2_pattern = re.compile(r'^.+CallData\s*count\((\d+)\)')

inv_type_wo_data = 'inv_wo_data'
inv_type_w_data = 'inv_w_data'
inv_type_count_0_wo_data = 'inv_count_0_wo_data'
inv_type_count_0_w_data = 'inv_count_0_w_data'
invoke_types = [ inv_type_wo_data, inv_type_w_data, inv_type_count_0_wo_data, inv_type_count_0_w_data ]

method_data_header_size = 216

force_process = True

mdo_delim = '------------------------------------------------------------------------'

def process_stdout(version: str, bench: str):
    result = BenchResult(version=version, bench=bench, total_mdo_size=0,
                         total_special_mdo_size=0, total_special_mdos=0,
                         total_invokes=0, total_bytecodes=0, time_median_ns=0, time_95_percentile_ns=0, time_99_percentile_ns=0)

    for type in invoke_types:
        result.invokes[type] = 0

    for type in [inv_type_count_0_w_data, inv_type_w_data]:
        result.invokes_mdo_sizes[type] = 0

    cached_path = f'cached/{get_stdout(version, bench)}.json'
    try:
        if not force_process:
            with open(cached_path) as f:
                data = json.load(f)
            return from_dict(BenchResult, data)
    except FileNotFoundError:
        pass

    with open(get_stdout(version, bench)) as f:
        f2 = open(f'{get_stdout(version, bench)}.non_invokes', 'w')
        end = False
        eof = False

        line = ''

        def readline():
            nonlocal line
            line = f.readline()
            if len(line) == 0:
                nonlocal eof
                eof = True
                return ''
            line = line[:-1]
            return line

        readline()
        while not eof:
            if line.startswith('Total MDO size:'):
                result.total_mdo_size = int(line.split(':')[1].split()[0])
                end = True
                readline()
                continue
            elif line.startswith(mdo_delim):
                name = readline()
                if name.startswith('Total MDO size:'):
                    continue

                if '(' not in name:
                    continue

                name = name.split('(')[0]

                while not line.startswith(mdo_delim):
                    m = invoke_pattern.match(line)

                    if m is not None:
                        if top_level_invoke_pattern.match(line) is not None:
                            result.total_invokes += 1
                            result.total_bytecodes += 1

                        readline()
                        m = invoke2_pattern.match(line)
                        if version == 'upstream' or m is None:
                            continue

                        count = int(m.group(1))
                        special_size = 0
                        has_special = False
                        while True:
                            readline()

                            m = special_size_pattern.match(line)
                            if m is not None:
                                special_size = int(m.group(1))
                                has_special = True
                                break

                            if bytecode_pattern.match(line) is not None:
                                break

                        if has_special:
                            result.total_special_mdos += 1
                            if count == 0:
                                result.invokes[inv_type_count_0_w_data] += 1
                                result.invokes_mdo_sizes[inv_type_count_0_w_data] += special_size
                            else:
                                result.invokes[inv_type_w_data] += 1
                                result.invokes_mdo_sizes[inv_type_w_data] += special_size
                        else:
                            if count == 0:
                                result.invokes[inv_type_count_0_wo_data] += 1
                            else:
                                result.invokes[inv_type_wo_data] += 1

                        continue

                    m = special_mdo_pattern.match(line)
                    if m is not None:
                        result.total_special_mdo_size += int(m.group(1))
                        readline()
                        continue

                    m = top_level_bytecode_pattern.match(line)
                    if m is not None:
                        result.total_bytecodes += 1
                        readline()
                        continue

                    readline()
                continue
            elif end:
                m = nmt_pattern.match(line)
                if m is not None:
                    commited = int(m.group(2))
                    if commited != 0:
                        result.memory_tracking[m.group(1)] = commited
                readline()
                continue
            else:
                readline()

        f2.close()

    with open(get_csv(version, bench)) as file:
        csv_reader = csv.reader(file, delimiter=',')
    
        durations = []
        first = True
        for row in csv_reader:
            if first:
                first = False
                continue
            durations.append(int(row[1]))
    
        durations.sort()
        result.time_median_ns = int(np.percentile(durations, 50))
        result.time_95_percentile_ns = int(np.percentile(durations, 95))
        result.time_99_percentile_ns = int(np.percentile(durations, 99))

    with open(cached_path, 'w') as f:
        json.dump(asdict(result), f, indent=4)

    return result

upstream_stdouts = [process_stdout(upstream, bench) for bench in bench_list]
latest_stdouts = [process_stdout(latest, bench) for bench in bench_list]

stdout_diffs = [BenchDiff(upstream, latest) for upstream, latest in zip(upstream_stdouts, latest_stdouts)]

with open("test", 'w') as f:
        json.dump([{"bench": t.upstream.bench, "time_median_rel": t.time_median_rel_diff, "time_95_percentile_rel": t.time_95_percentile_rel_diff, "time_99_percentile_rel": t.time_99_percentile_rel_diff} for t in stdout_diffs], f, indent=4)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_abs_rel(stdout_diffs: list[tuple[int, float]], title: str, legend: str, relative_unit: str, absolute_unit: str,
                 y_align_factor: float | None, figsize: tuple[int, int] | None = None):
    x = np.arange(bench_count)

    mdo_abs_diff = [x[0] for x in stdout_diffs]
    mdo_rel_diff = [x[1] * 100 for x in stdout_diffs]

    width = 0.4

    _, ax = plt.subplots(figsize=((16, 4) if figsize is None else figsize))

    ax.bar(x - width / 2, mdo_rel_diff, width=width, label=f'Relative ({relative_unit})', color="#74c476", capsize=4)
    ax.grid(axis='y', linestyle='--', alpha=0.5)

    ax2 = ax.twinx()

    ax2.bar(x + width / 2, mdo_abs_diff, width=width, label=f'Absolute{f" ({absolute_unit}" if absolute_unit else ""}', color="#eecb81", capsize=4)
    ax.grid(axis='y', linestyle='--', alpha=0.5)

    ax2.set_yticks([x * y_align_factor for x in ax.get_yticks()])
    ax2.set_ylim(ax.get_ylim()[0] * y_align_factor, ax.get_ylim()[1] * y_align_factor)

    ax.set_xticks(x)
    ax.set_xticklabels(bench_list, rotation=45, ha='right')

    handles1, labels1 = ax.get_legend_handles_labels()
    handles2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(handles1 + handles2, labels1 + labels2, title=legend, loc='upper right')
    ax.set_title(title)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

x = np.arange(bench_count)
width = 0.4

colors = [
    "#EE3B3B",
    "#F7A821",
    "#E4D939",
    "#69D34B",
    "#49E0DE",
    "#3B5BCD",
    "#BE2BD8",
    "#BF3E7A",
]

darker_colors = []
for color in colors:
    new_color = [int(int(x, base=16) * 0.6) for x in [color[1:3], color[3:5], color[5:7]]]
    darker_colors.append('#' + ''.join(hex(x)[2:].rjust(2, '0') for x in new_color))

array_types = 'abcdfils'
array_funcs = ['load', 'store']

width = 0.4

plot_abs_rel([(r.total_invokes, r.total_invokes / r.total_bytecodes) for r in latest_stdouts],
             'Invoke instructions (and relative to all instructions)', 'Invoke instructions',
             relative_unit='%', absolute_unit=None, y_align_factor=6000)

fig, ax = plt.subplots(figsize=(16, 6))

invokes_w_labels = {
    inv_type_wo_data: 'Invoke w/o inline (invoked)',
    inv_type_w_data: 'Invoke w/ inline (invoked)',
    inv_type_count_0_wo_data: 'Invoke w/o inline (no invocations)',
    inv_type_count_0_w_data: 'Invoke w/ inline (no invocations)',
}

bottoms = np.zeros(bench_count)
for i, (type, label) in enumerate(invokes_w_labels.items()):
    data = [r.invokes[type] for r in latest_stdouts]
    ax.bar(x, data, width=width, bottom=bottoms, label=label, color=colors[i], edgecolor='#323232')
    bottoms += data

ax.set_xticks(x)
ax.set_xticklabels(bench_list, rotation=45, ha='right')

ticks = ax.get_yticks()
ax.set_yticks(np.concatenate((ticks, np.arange(ticks[0], ticks[1], ticks[1] / 2))))
labels = ax.get_yticklabels()

ax.grid(axis='y', linestyle='--', alpha=0.5)

ax.legend(title="Invoke Types")

fig, ax = plt.subplots(figsize=(16, 6))

invokes_w_labels = {
    inv_type_w_data: 'Special data size (invoked)',
    inv_type_count_0_w_data: 'Special data size (no invocations)',
}

bottoms = np.zeros(bench_count)
for i, (type, label) in enumerate(invokes_w_labels.items()):
    data = [r.invokes_mdo_sizes[type] / 1024 for r in latest_stdouts]
    ax.bar(x, data, width=width, bottom=bottoms, label=label, color=colors[i * 2 + 1], edgecolor='#323232')
    bottoms += data

ax.set_xticks(x)
ax.set_xticklabels(bench_list, rotation=45, ha='right')

ticks = ax.get_yticks()
ax.set_yticks(np.concatenate((ticks, np.arange(ticks[0], ticks[1], ticks[1] / 2))))
labels = ax.get_yticklabels()

ax.grid(axis='y', linestyle='--', alpha=0.5)
ax.set_ylabel('Size (KB)')

ax.legend(title="Special data sizes")
# ax.set_title('Specialized MDO size')


plt.show()


In [ ]:
plot_abs_rel([(diff.total_mdo_size_abs_diff / 1024, diff.total_mdo_size_rel_diff - 1) for diff in stdout_diffs],
             'Total MDO size increase', 'MDO size increase', '%', 'KB', y_align_factor=150, figsize=(16, 4))

plot_abs_rel([(diff.total_non_special_mdo_size_abs_diff / 1024, diff.total_non_special_mdo_size_rel_diff - 1) for diff in stdout_diffs],
             'Total non-special MDO size increase', 'Non-special MDO size increase', '%', 'KB', y_align_factor=150, figsize=(16, 4))

plot_abs_rel([(d.memory_tracking_abs_diff['Metaspace'] / 1024, d.memory_tracking_rel_diff['Metaspace'] - 1) for d in stdout_diffs],
        'Metaspace commited increase', 'Metaspace commited size increase', '%', 'KB', y_align_factor=640, figsize=(16, 4))


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

x = np.arange(bench_count)
width = 0.4

colors = [
    "#EE3B3B",
    "#F7A821",
    "#E4D939",
    "#69D34B",
    "#49E0DE",
    "#3B5BCD",
    "#BE2BD8",
    "#BF3E7A",
]

darker_colors = []
for color in colors:
    new_color = [int(int(x, base=16) * 0.6) for x in [color[1:3], color[3:5], color[5:7]]]
    darker_colors.append('#' + ''.join(hex(x)[2:].rjust(2, '0') for x in new_color))

width = 0.4

data_only = [(r.total_special_mdo_size - (r.total_special_mdos * method_data_header_size)) for r in latest_stdouts]

header_only = [r.total_special_mdos * method_data_header_size for r in latest_stdouts]

old_mdo_size = [r.total_mdo_size for r in upstream_stdouts]
new_mdo_size = [total - head for total, head in zip([r.total_mdo_size for r in latest_stdouts], header_only)]

new_abs_diff = [l - u for u, l in zip(old_mdo_size, new_mdo_size)]
new_rel_diff = [l / u for u, l in zip(old_mdo_size, new_mdo_size)]

plot_abs_rel([(abs / 1024, rel - 1) for abs, rel in zip(new_abs_diff, new_rel_diff)],
             'Projected MDO size increase', 'MDO size increase', '%', 'KB', y_align_factor=150, figsize=(16, 4))


data_only = [t / 1024 for t in data_only]
header_only = [t / 1024 for t in header_only]

fig, ax = plt.subplots(figsize=(16, 6))

data_types = ['Header', 'ProfileData']

bottoms = np.zeros(bench_count)
for i, type in enumerate(data_types):
    data = [header_only, data_only][i]
    ax.bar(x, data, width=width, bottom=bottoms, label=type, color=colors[i], edgecolor='#323232')
    bottoms += data

ax.set_xticks(x)
ax.set_xticklabels(bench_list, rotation=45, ha='right')

ticks = ax.get_yticks()
ax.set_yticks(np.concatenate((ticks, np.arange(ticks[0], ticks[1], ticks[1] / 2))))
labels = ax.get_yticklabels()

ax.grid(axis='y', linestyle='--', alpha=0.5)

ax.legend(title="Specialized MDO")
ax.set_title('Specialized MDO size')

plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

x = np.arange(bench_count)

width = 0.4

data = [r.total_special_mdo_size for r in latest_stdouts]

plot_abs_rel([(r.total_special_mdo_size / 1024, (r.total_special_mdo_size / (r.total_mdo_size - r.total_special_mdo_size)) - 1) for r in latest_stdouts],
        'Specialized', 'Specialized relative to total', '%', 'KB', y_align_factor=240, figsize=(16, 4))
